# Tabular Parsing

In [1]:
import os
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

src_dir = Path.cwd() / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from lake_agent.indexing.tabular import DeterministicTabularParser

In [2]:
data_dir = Path(os.environ["DATALAKE_DIR"]).expanduser().resolve()
parser = DeterministicTabularParser()

files = sorted(
    path for path in data_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in {".csv", ".tsv", ".xlsx"}
)

print("DATALAKE_DIR:", data_dir)
print("Num files:", len(files))

DATALAKE_DIR: /home/thanhvinh/Project/LakeAgent/data/Data-Lake
Num files: 22


In [3]:
for file_path in files:
    relative_path = file_path.relative_to(data_dir).as_posix()
    print("\n===", relative_path, "===")

    try:
        result = parser.parse_file(file_path, relative_path=relative_path)
    except Exception as exc:
        print("Parse error:", exc)
        continue

    print("Format:", result.file_format)
    print("Table len:", len(result.tables))
    if result.parse_warnings:
        print("Parse warnings:", result.parse_warnings)

    for table in result.tables:
        print("\nTable:", table.table_name)
        print("Sheet:", table.sheet_name)
        print("Header row:", table.header_row_index)
        print("Raw header:", table.raw_header)
        print("Columns:", [column.name for column in table.columns])
        print("Row count:", table.row_count)
        if table.warnings:
            print("Warnings:", table.warnings)
        print("Preview rows:")
        for row in table.preview_rows[:3]:
            print(row)


=== archeology/climateMeasurements.xlsx ===
Format: xlsx
Table len: 1

Table: Sheet1
Sheet: Sheet1
Header row: 3
Raw header: ['Site', 'Hole', 'Core', 'Section', 'Interval_cm', 'Depth below seafloor_m', 'Splice depth_m', 'Age_ky', 'Al', 'Si', 'S', 'K', 'Ca', 'Ti', 'Mn', 'Fe', 'Rb', 'Sr', 'Zr', 'Ba', '', 'Age_ky', 'PC1', 'PC2', '', 'Age_ky', 'ODP 967 Dust proxy', '', 'Age_ky', 'ODP 967 wet-dry index']
Columns: ['Site', 'Hole', 'Core', 'Section', 'Interval_cm', 'Depth below seafloor_m', 'Splice depth_m', 'Age_ky', 'Al', 'Si', 'S', 'K', 'Ca', 'Ti', 'Mn', 'Fe', 'Rb', 'Sr', 'Zr', 'Ba', 'column_21', 'Age_ky_2', 'PC1', 'PC2', 'column_25', 'Age_ky_3', 'ODP 967 Dust proxy', 'column_28', 'Age_ky_4', 'ODP 967 wet-dry index']
Row count: 8360
Warnings: ['Skipped 3 leading row(s) before the detected header.']
Preview rows:
['ODP 967', 'D', '1', '1', '4', '0.04', '0.04', '0.22825100000000001', '64819.915275048799', '126450.92007159699', '1977.6335915260199', '10665.8132794049', '90508.756301781206', 